# DQ Analysis — BDR Readiness Dataset
quick exploratory pass to understand what's broken before scoring

starting with raw files, going issue by issue as i find them

In [133]:
import pandas as pd
import numpy as np
from pathlib import Path

In [134]:


RAW = Path('../data/raw')

leads = pd.read_csv(RAW / 'leads.csv')
print ('leads done')
contacts = pd.read_csv(RAW / 'contacts.csv')
print ('contacts done')

leads done
contacts done


In [135]:
leads.shape

(600, 23)

In [136]:
accounts = pd.read_csv(RAW / 'accounts.csv')
print ('accounts done')

accounts done


In [137]:

campaigns = pd.read_csv(RAW / 'campaigns.csv')
print ('campaigns done')

campaigns done


In [138]:
cm = pd.read_csv(RAW / 'campaign_members.csv')
print ('campaign_members done')

campaign_members done


In [139]:
leads.shape, contacts.shape, accounts.shape, campaigns.shape, cm.shape

((600, 23), (400, 21), (200, 8), (60, 4), (4418, 8))

In [8]:
leads.head(3)

,lead_id,first_name,last_name,email,title,job_level,job_persona,phone,account_id,industry,...,email_opt_out,email_bounced,no_longer_with_company,is_converted,converted_contact_id,true_persona,free_email_known,actual_free_email,mkto_lead_score,_archetype
0,L00001,Rebecca,Young,ryoung@gravitasgroup.com,Manager of Something,Manager,NaN,+1-555-9070,ACC0047,Telecommunications,...,False,False,False,False,NaN,Partner,False,False,42,non_prospect
1,L00002,Jacob,Roberts,jroberts@glacierlabs.com,Individual Contributor of Something,Individual Contributor,NaN,+1-555-3742,ACC0120,Manufacturing,...,False,False,False,True,NaN,Prospect,False,False,33,researcher
2,L00003,Dorothy,White,dwhite@heliostechnologies.com,NaN,Individual Contributor,NaN,NaN,ACC0161,Manufacturing,...,True,False,False,True,C00120,Employee,False,False,46,non_prospect


In [ ]:
contacts.head(3)

---
## DQ-4 — created_date looks suspicious
noticed a lot of the same date while scrolling. let me check

In [79]:
leads['created_date'].value_counts().head(10)

created_date
2024-01-01    496
2023-01-21      2
2023-12-04      2
2022-03-29      2
2022-02-04      2
2022-12-05      2
2022-06-18      2
2022-01-08      1
2023-12-30      1
2022-04-01      1
Name: count, dtype: int64

In [11]:
# we can see that more than 80% leads have same ETL date which must be the load dat

In [80]:
contacts['created_date'].value_counts().head(10)

created_date
2024-01-01    140
2022-07-13      4
2022-09-08      3
2022-07-16      3
2023-11-14      2
2023-04-30      2
2023-05-23      2
2022-04-19      2
2022-03-07      2
2023-01-17      2
Name: count, dtype: int64

In [42]:
contacts.isna().sum()

contact_id                0
first_name                0
last_name                 0
email                     0
title                    51
job_level                 0
job_persona             158
phone                    65
account_id               60
industry                 60
employee_count           60
created_date              0
is_mql                    0
mql_date                265
email_opt_out             0
email_bounced             0
is_converted              0
primary_lead_id         236
true_persona              0
free_email_known          0
actual_free_email         0
mkto_contact_score_c      0
_archetype                0
dtype: int64

In [31]:
# Same issue for contacts as well. We can ignore created_date for both leads and contacts as it is not the actual created date but the load date.

In [81]:

etl_date = '2024-01-01'

pct_leads_etl = (leads['created_date'] == etl_date).mean()
pct_contacts_etl = (contacts['created_date'] == etl_date).mean()

print(f'leads with ETL date: {pct_leads_etl:.1%}')
print(f'contacts with ETL date: {pct_contacts_etl:.1%}')

leads with ETL date: 82.7%
contacts with ETL date: 35.0%


In [82]:
# 80% of leads — this date is completely useless for sequencing or recency logic
# contacts are better (34%) but still bad
# should NOT use created_date for anything in scoring

leads['created_date'].value_counts().nlargest(5).plot(kind='bar', title='Lead created_date distribution (top 5)')

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

---
## DQ-2 — Duplicate emails
leads and contacts might share emails, or have internal dupes. let me check both

In [83]:
# dupes within leads
lead_email_counts = leads['email'].value_counts()
print('leads with duplicate email:')
print(lead_email_counts[lead_email_counts > 1].head(10))

leads with duplicate email:
email
newsletter@marketingblast.com    13
leads@databroker.io              11
sales@megacorp.com                6
info@b2bleads.net                 6
marketing@globalpartners.com      6
contact@vendorlist.net            5
info@techconf.com                 5
sales@cloudvendor.io              5
admin@tradeshow.events            4
sales@enterprisetech.com          3
Name: count, dtype: int64


In [84]:
leads['lead_id'].nunique(), leads['email'].nunique() 

(600, 523)

In [85]:
duplicate_emails

email
aanderson@umbragroup.com           2
admin@tradeshow.events             4
ajohnson@atlascorp.com             2
arodriguez@obsidiangroup.com       2
atorres@junctioncorp.com           2
awright@gravitasdynamics.com       2
contact@vendorlist.net             5
ctorres@quantumnetworks.com        2
hello@sdrlists.com                 3
info@b2bleads.net                  6
info@securitysummit.org            2
info@techconf.com                  5
jadams@nexustechnologies.com       2
jcampbell@unknown.com              2
jlewis@luminarygroup.com           2
jlopez@unknown.com                 2
jnelson@heliostechnologies.com     2
jroberts@meridianventures.com      2
leads@databroker.io               11
lwilson@orbitsolutions.com         2
mallen@fusionlabs.com              2
marketing@globalpartners.com       6
michael.clark@outlook.com          2
mramirez@dynexsystems.com          2
mtorres@novanetworks.com           2
newsletter@marketingblast.com     13
nramirez@fusionlabs.com         

In [86]:
# count of emails with multiple lead_ids
email_lead_counts = leads.groupby('email')['lead_id'].nunique()
duplicate_emails = email_lead_counts[email_lead_counts > 1]
print('Emails associated with multiple lead_ids:', len(duplicate_emails))

Emails associated with multiple lead_ids: 32


In [87]:
# dupes within contacts
contact_email_counts = contacts['email'].value_counts()
print('contacts with duplicate email:')
print(contact_email_counts[contact_email_counts > 1])

contacts with duplicate email:
email
noreply@formspam.net             14
subscribe@vendorblast.com        10
hello@sdrlists.com                3
sales@enterprisetech.com          3
info@securitysummit.org           3
ctaylor@obsidiangroup.com         2
info@techconf.com                 2
rcampbell@unknown.com             2
krobinson@atlaspartners.com       2
mwilliams@unknown.com             2
cwalker@aethersolutions.com       2
jmartinez@paragondynamics.com     2
Name: count, dtype: int64


In [88]:
# cross-object dupes — same email on a lead AND a contact (the conversion case)
all_emails = pd.concat([
    leads[['email']].assign(source='lead'),
    contacts[['email']].assign(source='contact')
])

cross_dupes = all_emails.groupby('email')['source'].nunique()
print(f'emails appearing in both leads AND contacts: {(cross_dupes > 1).sum()}')

emails appearing in both leads AND contacts: 40


In [89]:
# also spot the shared/spam addresses
all_emails_combined = list(leads['email']) + list(contacts['email'])
email_freq = pd.Series(all_emails_combined).value_counts()
print('highest frequency emails (spam/shared):')
print(email_freq[email_freq >= 3].head(10))

highest frequency emails (spam/shared):
noreply@formspam.net             14
newsletter@marketingblast.com    13
leads@databroker.io              11
subscribe@vendorblast.com        10
sales@megacorp.com                7
info@techconf.com                 7
info@b2bleads.net                 7
sales@cloudvendor.io              6
sales@enterprisetech.com          6
marketing@globalpartners.com      6
Name: count, dtype: int64


---
## DQ-6 — Non-prospect contamination + null job_persona
the `true_persona` field tells us who's actually a prospect vs junk

In [104]:
leads.columns

Index(['lead_id', 'first_name', 'last_name', 'email', 'title', 'job_level',
       'job_persona', 'phone', 'account_id', 'industry', 'employee_count',
       'created_date', 'lead_status', 'is_current_mql', 'mql_date',
       'email_opt_out', 'email_bounced', 'no_longer_with_company',
       'is_converted', 'converted_contact_id', 'free_email_known',
       'actual_free_email', 'mkto_lead_score'],
      dtype='str')

In [105]:
leads['job_level'].isna().sum()

np.int64(74)

In [106]:
leads['job_persona'].isna().sum()

np.int64(249)

In [107]:
leads['job_persona'].value_counts()

job_persona
Non-Prospect       106
Technical Buyer     86
Financial Buyer     67
Influencer          56
CISO                36
Name: count, dtype: int64

In [109]:
leads['job_level'].value_counts()

job_level
Individual Contributor    203
Manager                   135
Director                  108
VP                         60
C-Level                    20
Name: count, dtype: int64

In [111]:
contacts['job_persona'].isna().sum()

np.int64(152)

In [112]:
contacts['job_persona'].value_counts()

job_persona
Non-Prospect       71
Technical Buyer    60
Financial Buyer    56
Influencer         38
CISO               23
Name: count, dtype: int64

In [ ]:
l['job_persona'].value_counts()

In [ ]:
# so roughly 38% are non-prospects (partner/competitor/employee/vendor)
# that's a lot of noise in the scoring pool

non_prospect_leads = leads[leads['true_persona'] != 'Prospect']
print(f'non-prospect leads: {len(non_prospect_leads)} ({len(non_prospect_leads)/len(leads):.1%})')

In [ ]:
# now job_persona nulls — this is a separate issue from true_persona
print('job_persona null rate in leads:', leads['job_persona'].isna().mean().round(3))
print('job_persona null rate in contacts:', contacts['job_persona'].isna().mean().round(3))

In [ ]:
# ~40% null — way too high to just drop. need to impute or handle carefully
# but NOT with global mean — leads/contacts have different distributions

print('job_persona distribution in leads (non-null):')
print(leads['job_persona'].value_counts(dropna=True))
print()
print('job_persona distribution in contacts (non-null):')
print(contacts['job_persona'].value_counts(dropna=True))

---
## DQ-7 — Missing fields
quick null scan across both tables

In [120]:
key_lead_cols = ['email', 'title', 'phone', 'account_id', 'job_level', 'job_persona', 'industry']
key_contact_cols = ['email', 'title', 'phone', 'account_id', 'job_level', 'job_persona', 'industry']

print('null % — leads')
print(leads[key_lead_cols].isna().mean().mul(100).round(1).sort_values(ascending=False))
print()
print('null % — contacts')
print(contacts[key_contact_cols].isna().mean().mul(100).round(1).sort_values(ascending=False))

null % — leads
job_persona    41.5
phone          18.7
account_id     15.3
industry       15.3
job_level      12.3
title          10.3
email           0.0
dtype: float64

null % — contacts
job_persona    38.0
phone          18.2
job_level      16.8
account_id     15.0
industry       15.0
title          11.0
email           0.0
dtype: float64


In [121]:
# account_id missing means we lose industry + employee_count too — it cascades
no_account_leads = leads['account_id'].isna().sum()
no_account_contacts = contacts['account_id'].isna().sum()
print(f'leads with no account: {no_account_leads} ({no_account_leads/len(leads):.1%})')
print(f'contacts with no account: {no_account_contacts} ({no_account_contacts/len(contacts):.1%})')

leads with no account: 92 (15.3%)
contacts with no account: 60 (15.0%)


In [122]:
# how many leads have no account AND no industry independently
leads['has_any_company_signal'] = leads['account_id'].notna() | leads['industry'].notna()
print('leads with zero company signal:', (~leads['has_any_company_signal']).sum())

leads with zero company signal: 92


---
## DQ-8 — Automation-inflated engagement
some records look very active but it's all 'Sent' status — automated blasts, not real intent

In [123]:
# status distribution overall
print('campaign member_status breakdown:')
print(cm['member_status'].value_counts())
print()
print(f"'Sent' share of all events: {(cm['member_status']=='Sent').mean():.1%}")

campaign member_status breakdown:
member_status
Sent          3060
Clicked        609
Responded      316
Registered     169
Attended       167
No Show         97
Name: count, dtype: int64

'Sent' share of all events: 69.3%


In [124]:
# per-person automation share
person_cm = cm.groupby('entity_id').agg(
    total=('cm_id', 'count'),
    sent=('member_status', lambda x: (x == 'Sent').sum())
).assign(auto_share=lambda df: df['sent'] / df['total'])

print('automation share distribution:')
print(person_cm['auto_share'].describe().round(3))
print()
print(f"records where >70% of events are 'Sent': {(person_cm['auto_share'] > 0.7).sum()} ({(person_cm['auto_share'] > 0.7).mean():.1%})")

automation share distribution:
count    759.000
mean       0.500
std        0.392
min        0.000
25%        0.000
50%        0.500
75%        0.875
max        1.000
Name: auto_share, dtype: float64

records where >70% of events are 'Sent': 292 (38.5%)


In [126]:
person_cm.sort_values('auto_share', ascending=False).head(10)

,total,sent,auto_share
entity_id,,,
C00005,1,1,1.0
L00594,9,9,1.0
L00019,1,1,1.0
L00012,2,2,1.0
C00399,15,15,1.0
L00002,11,11,1.0
L00025,8,8,1.0
C00383,1,1,1.0
L00078,1,1,1.0


In [127]:
# look at the most inflated records
inflated = person_cm[person_cm['auto_share'] > 0.7].sort_values('total', ascending=False)
print('top 10 most inflated records:')
print(inflated.head(10))

top 10 most inflated records:
           total  sent  auto_share
entity_id                         
L00432        29    25    0.862069
C00262        29    25    0.862069
L00427        29    25    0.862069
L00106        29    25    0.862069
L00298        29    25    0.862069
C00042        29    25    0.862069
C00139        17    17    1.000000
C00194        17    15    0.882353
L00183        17    15    0.882353
L00464        17    16    0.941176


In [128]:
# these folks have 10-17 events but almost all 'Sent' — would score very high on naive count
# genuine events per person distribution (excluding Sent)
genuine = cm[cm['member_status'] != 'Sent']
genuine_per_person = genuine.groupby('entity_id').size()

print('genuine (non-Sent) engagement events per person:')
print(genuine_per_person.describe().round(2))
print()
print('people with 0 genuine events:', len(person_cm) - len(genuine_per_person))

genuine (non-Sent) engagement events per person:
count    626.00
mean       2.17
std        1.75
min        1.00
25%        1.00
50%        2.00
75%        2.00
max       10.00
dtype: float64

people with 0 genuine events: 133


---
## DQ-9 — Opted out / bounced / no longer with company
these are hard blockers — can't outreach regardless of score

In [140]:
for col in ['email_opt_out', 'email_bounced']:
    n_leads = leads[col].sum()
    n_contacts = contacts[col].sum()
    print(f'{col}: leads={n_leads} ({n_leads/len(leads):.1%}), contacts={n_contacts} ({n_contacts/len(contacts):.1%})')

email_opt_out: leads=138 (23.0%), contacts=91 (22.8%)
email_bounced: leads=71 (11.8%), contacts=33 (8.2%)


In [141]:
# 'no_longer_with_company' only on leads in this dataset
if 'no_longer_with_company' in leads.columns:
    n = leads['no_longer_with_company'].sum()
    print(f'no_longer_with_company: {n} leads ({n/len(leads):.1%})')

no_longer_with_company: 51 leads (8.5%)


In [142]:
# how many are blocked on MULTIPLE criteria — stacking
leads['blocker_count'] = (
    leads['email_opt_out'].astype(int) +
    leads['email_bounced'].astype(int) +
    leads.get('no_longer_with_company', pd.Series(0, index=leads.index)).astype(int)
)
print('leads by blocker count:')
print(leads['blocker_count'].value_counts().sort_index())

leads by blocker count:
blocker_count
0    372
1    197
2     30
3      1
Name: count, dtype: int64


In [143]:
# check if opted-out records have high engagement scores — would be a problem if we scored without filtering
opted_out_lead_ids = leads[leads['email_opt_out'] == True]['lead_id'].tolist()
opted_out_cm = cm[cm['entity_id'].isin(opted_out_lead_ids)]
print(f'opted-out leads with campaign activity: {opted_out_cm["entity_id"].nunique()}')
print('their event counts:')
print(opted_out_cm.groupby('entity_id').size().describe().round(1))

opted-out leads with campaign activity: 111
their event counts:
count    111.0
mean       5.8
std        5.5
min        1.0
25%        2.0
50%        3.0
75%        9.5
max       29.0
dtype: float64


---
## DQ-1 — Broken conversion links
converted leads that have no matching contact_id — orphaned conversions

In [144]:
converted = leads[leads['is_converted'] == True]
print(f'total converted leads: {len(converted)}')
print(f'of those, missing converted_contact_id: {converted["converted_contact_id"].isna().sum()}')
print(f'broken link rate: {converted["converted_contact_id"].isna().mean():.1%}')

total converted leads: 200
of those, missing converted_contact_id: 37
broken link rate: 18.5%


In [145]:
# even for the ones that DO have a contact_id, does the contact actually exist?
linked_contact_ids = converted['converted_contact_id'].dropna().tolist()
existing_contact_ids = set(contacts['contact_id'])

dangling = [cid for cid in linked_contact_ids if cid not in existing_contact_ids]
print(f'converted leads pointing to non-existent contacts: {len(dangling)}')

converted leads pointing to non-existent contacts: 0


In [146]:
# the reverse — contacts that claim to have a lead origin
if 'primary_lead_id' in contacts.columns:
    has_origin = contacts['primary_lead_id'].notna()
    print(f'contacts with a primary_lead_id: {has_origin.sum()}')
    # do those leads still exist?
    existing_lead_ids = set(leads['lead_id'])
    dangling_contacts = contacts.loc[has_origin, 'primary_lead_id'].apply(lambda x: x not in existing_lead_ids).sum()
    print(f'of those, pointing to missing leads: {dangling_contacts}')

contacts with a primary_lead_id: 163
of those, pointing to missing leads: 0


---
## DQ-3 — MQL date integrity
mql_date should predate or equal conversion. let me check if that holds

In [147]:
mql_leads = leads[leads['is_current_mql'] == True].copy()
print(f'MQL leads: {len(mql_leads)}')
print(f'of those, null mql_date: {mql_leads["mql_date"].isna().sum()}')

MQL leads: 161
of those, null mql_date: 0


In [148]:
# for converted MQL leads, mql_date often gets overwritten with conversion date
# check if mql_date clustering around conversion period
converted_mql = leads[(leads['is_converted'] == True) & (leads['is_current_mql'] == True)].copy()
print(f'leads that are both converted AND MQL: {len(converted_mql)}')
if len(converted_mql) > 0:
    print('\nmql_date distribution for converted MQLs:')
    print(pd.to_datetime(converted_mql['mql_date']).dt.year.value_counts())

leads that are both converted AND MQL: 50

mql_date distribution for converted MQLs:
mql_date
2023    50
Name: count, dtype: int64


In [149]:
# mql_date spread for regular (non-converted) MQLs
non_converted_mql = leads[(leads['is_converted'] == False) & (leads['is_current_mql'] == True)].copy()
print('mql_date distribution for non-converted MQLs:')
print(pd.to_datetime(non_converted_mql['mql_date']).dt.year.value_counts())

mql_date distribution for non-converted MQLs:
mql_date
2022    64
2023    47
Name: count, dtype: int64


---
## DQ-5 — Competing score fields (Marketo)
leads have `mkto_lead_score` (int), contacts have `mkto_contact_score_c` (float) — different field names, different dtypes

In [150]:
print('mkto_lead_score dtype:', leads['mkto_lead_score'].dtype)
print(leads['mkto_lead_score'].describe().round(1))
print()
print('mkto_contact_score_c dtype:', contacts['mkto_contact_score_c'].dtype)
print(contacts['mkto_contact_score_c'].describe().round(1))

mkto_lead_score dtype: int64
count    600.0
mean      44.3
std       31.4
min        0.0
25%       14.0
50%       46.0
75%       71.0
max       99.0
Name: mkto_lead_score, dtype: float64

mkto_contact_score_c dtype: float64
count    400.0
mean      47.4
std       31.9
min        0.0
25%       17.3
50%       49.0
75%       75.2
max       99.9
Name: mkto_contact_score_c, dtype: float64


In [151]:
# DQ-10: score resets — check zero inflation
print(f'leads with mkto score = 0: {(leads["mkto_lead_score"] == 0).sum()} ({(leads["mkto_lead_score"] == 0).mean():.1%})')
print(f'contacts with mkto score = 0: {(contacts["mkto_contact_score_c"] == 0).sum()} ({(contacts["mkto_contact_score_c"] == 0).mean():.1%})')

leads with mkto score = 0: 67 (11.2%)
contacts with mkto score = 0: 39 (9.8%)


In [ ]:
# score distribution — are these even comparable?
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
leads['mkto_lead_score'].plot(kind='hist', bins=20, ax=axes[0], title='mkto_lead_score (leads)', color='steelblue')
contacts['mkto_contact_score_c'].plot(kind='hist', bins=20, ax=axes[1], title='mkto_contact_score_c (contacts)', color='coral')
plt.tight_layout()
plt.show()
# they look roughly similar but zero-inflated — the reset problem

---
## DQ-11 — Free email leakage
the `free_email_known` flag should detect free domains (gmail, yahoo etc.) but it misses some

In [ ]:
FREE_DOMAINS = {'gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'protonmail.com'}

leads['actual_free_detected'] = leads['email'].apply(lambda e: str(e).split('@')[-1].lower() in FREE_DOMAINS)
contacts['actual_free_detected'] = contacts['email'].apply(lambda e: str(e).split('@')[-1].lower() in FREE_DOMAINS)

print('=== leads ===')
print('actually free email:  ', leads['actual_free_detected'].sum())
print('flagged as free (known):', leads['free_email_known'].sum())
print('missed (free but not flagged):', (leads['actual_free_detected'] & ~leads['free_email_known']).sum())

In [ ]:
print('=== contacts ===')
print('actually free email:  ', contacts['actual_free_detected'].sum())
print('flagged as free (known):', contacts['free_email_known'].sum())
print('missed (free but not flagged):', (contacts['actual_free_detected'] & ~contacts['free_email_known']).sum())

In [ ]:
# see which domains are slipping through
missed_leads = leads[leads['actual_free_detected'] & ~leads['free_email_known']]
print('missed free-email domains in leads:')
print(missed_leads['email'].apply(lambda e: e.split('@')[-1]).value_counts())

---
## DQ quick summary — how many records are affected by each issue

In [152]:
all_people = pd.concat([
    leads.assign(entity_type='Lead', record_id=leads['lead_id']),
    contacts.assign(entity_type='Contact', record_id=contacts['contact_id'])
], ignore_index=True)

total = len(all_people)

dq_summary = [
    ('DQ-1  broken conversion link',   converted['converted_contact_id'].isna().sum()),
    ('DQ-2  duplicate email',          all_people['email'].duplicated(keep=False).sum()),
    ('DQ-4  ETL created_date',         (all_people['created_date'] == etl_date).sum()),
    ('DQ-5  score field = 0 (reset)',  int(leads['mkto_lead_score'].eq(0).sum()) + int(contacts['mkto_contact_score_c'].eq(0).sum())),
    ('DQ-6  non-prospect in pool',     (all_people['true_persona'] != 'Prospect').sum()),
    ('DQ-6  null job_persona',         all_people['job_persona'].isna().sum()),
    ('DQ-7  missing account_id',       all_people['account_id'].isna().sum()),
    ('DQ-8  automation-inflated >70%', int((person_cm['auto_share'] > 0.7).sum())),
    ('DQ-9  opt-out or bounced',       int(all_people['email_opt_out'].sum() + all_people['email_bounced'].sum())),
    ('DQ-11 free email missed',        int((leads['actual_free_detected'] & ~leads['free_email_known']).sum() +
                                          (contacts['actual_free_detected'] & ~contacts['free_email_known']).sum())),
]

df_summary = pd.DataFrame(dq_summary, columns=['Issue', 'Affected Records'])
df_summary['% of Total'] = (df_summary['Affected Records'] / total * 100).round(1).astype(str) + '%'
df_summary

KeyError: 'true_persona'

---
## a few extra checks I want to do before closing
just random things that caught my eye

In [ ]:
# are leads and contacts going to the same campaigns? i.e. is campaign_members mixed
print('entity types in campaign_members:')
print(cm['entity_type'].value_counts())

In [ ]:
# any campaign_members with entity_id that doesn't exist in either leads or contacts
all_ids = set(leads['lead_id']) | set(contacts['contact_id'])
orphan_cm = cm[~cm['entity_id'].isin(all_ids)]
print(f'campaign_members with no matching person: {len(orphan_cm)}')

In [ ]:
# how many people have ZERO campaign history at all
people_with_cm = set(cm['entity_id'])
lead_no_cm = (~leads['lead_id'].isin(people_with_cm)).sum()
contact_no_cm = (~contacts['contact_id'].isin(people_with_cm)).sum()
print(f'leads with no campaign history: {lead_no_cm} ({lead_no_cm/len(leads):.1%})')
print(f'contacts with no campaign history: {contact_no_cm} ({contact_no_cm/len(contacts):.1%})')
# these will all get engagement_score=0 — valid but worth noting for the scoring model

In [ ]:
# named accounts — do they skew industry?
named_accs = accounts[accounts['is_named_account'] == True]
print('named account industry distribution:')
print(named_accs['industry'].value_counts())

In [ ]:
# also check account DNC — how many leads/contacts are at DNC accounts
dnc_account_ids = set(accounts[accounts['account_do_not_contact'] == True]['account_id'])
leads_at_dnc = leads['account_id'].isin(dnc_account_ids).sum()
contacts_at_dnc = contacts['account_id'].isin(dnc_account_ids).sum()
print(f'leads at DNC accounts: {leads_at_dnc}')
print(f'contacts at DNC accounts: {contacts_at_dnc}')